In [0]:

# Note: azure-eventhub is retained for Event Hub management utilities.
# The streaming read uses the Kafka protocol (no SDK dependency at runtime).
%pip install azure-eventhub -q

In [0]:
dbutils.library.restartPython()

In [0]:
# ============================================================
# CONFIGURATION
# ============================================================
# Best practice: Store secrets in Databricks Secret Scopes
# via dbutils.secrets.get(scope, key). Secrets are fetched at
# runtime from the configured secret scope — no hardcoded values.
#
# To set up secrets in your workspace:
#   1. databricks secrets create-scope --scope lakehouse-secrets
#   2. databricks secrets put --scope lakehouse-secrets --key event-hub-connection-string
#   3. databricks secrets put --scope lakehouse-secrets --key storage-account-key
#   4. databricks secrets put --scope lakehouse-secrets --key event-hub-namespace-sas-key
#
# Fallback to environment variables for local dev / CI testing.
# ============================================================

import os

# --- Secret Scope (preferred for production) ---
SECRET_SCOPE = "lakehouse-secrets"


def get_secret(key, default=None):
    """Fetch secret from Databricks Secret Scope, falling back to env var."""
    try:
        return dbutils.secrets.get(scope=SECRET_SCOPE, key=key)
    except Exception:
        return os.getenv(key.upper().replace("-", "_"), default)


# --- Event Hubs / Kafka settings ---
eh_namespace = get_secret("event-hub-namespace", "your-namespace")
eh_name = get_secret("event-hub-name", "your-event-hub")
eh_connection_string = get_secret("event-hub-connection-string", "")

kafka_bootstrap_servers = f"{eh_namespace}.servicebus.windows.net:9093"

eh_sasl = (
    'kafkashaded.org.apache.kafka.common.security.plain.'
    'PlainLoginModule required '
    'username="$ConnectionString" '
    f'password="{eh_connection_string}";'
)

# --- Azure Data Lake Storage settings ---
storage_account = os.getenv("AZURE_STORAGE_ACCOUNT", "your-storage-account")
storage_key = get_secret("storage-account-key", "")

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

# --- Path constants (single source of truth) ---
BRONZE_PATH = f"abfss://bronze@{storage_account}.dfs.core.windows.net/crypto_trades"
CHECKPOINT_PATH = f"abfss://checkpoint@{storage_account}.dfs.core.windows.net/bronze_crypto_checkpoint"

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType
)

# ============================================================
# SCHEMA DEFINITION — Future-proof & Fault-tolerant
# ============================================================
# 1. All fields are nullable (True) to tolerate upstream schema
#    evolution. If the producer adds/removes fields, existing
#    records won't fail deserialization.
# 2. The special "_corrupt_record" column captures any JSON message
#    that cannot be parsed against this schema, enabling downstream
#    investigation without dropping data or failing the stream.
# 3. New fields from the producer will appear as nulls until the
#    schema is explicitly extended here.
# ============================================================

market_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("asset_class", StringType(), True),
    StructField("trade_id", LongType(), True),
    StructField("trade_price", DoubleType(), True),
    StructField("trade_quantity", DoubleType(), True),
    StructField("event_type", StringType(), True),
    StructField("trade_timestamp", LongType(), True),
    StructField("_source", StringType(), True),
    StructField("_loaded_at", StringType(), True),
    StructField("_batch_id", StringType(), True),
    # Captures malformed JSON so the stream never fails
    StructField("_corrupt_record", StringType(), True),
])

In [0]:
# ============================================================
# STREAM READ — Azure Event Hubs via Kafka protocol
# ============================================================
# Notes:
# • "startingOffsets" = "latest" applies only on FIRST start;
#   subsequent restarts resume from the checkpoint automatically.
# • maxOffsetsPerTrigger throttles micro-batch size to prevent
#   OOM on sudden traffic spikes (back-pressure guard).
# • failOnDataLoss = "false" allows the stream to continue if
#   Event Hubs trims old events beyond retention window.
# • Increased request/session timeouts for transient network issues.
# ============================================================

raw_stream_df = (
    spark.readStream
         .format("kafka")
         .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
         .option("subscribe", eh_name)
         .option("kafka.security.protocol", "SASL_SSL")
         .option("kafka.sasl.mechanism", "PLAIN")
         .option("kafka.sasl.jaas.config", eh_sasl)
         .option("startingOffsets", "latest")
         .option("maxOffsetsPerTrigger", 50000)        # back-pressure guard
         .option("failOnDataLoss", "false")            # tolerate retention trim
         .option("kafka.request.timeout.ms", "60000")  # network resilience
         .option("kafka.session.timeout.ms", "30000")
         .load()
)

In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp

# ============================================================
# BRONZE TRANSFORMATION
# ============================================================
# Improvements:
# 1. Preserves raw Kafka value as `_raw_payload` for replay/debugging.
# 2. Parses JSON with PERMISSIVE mode — corrupt records are captured
#    in `_corrupt_record` instead of failing the stream.
# 3. Adds `_ingestion_ts` — the Databricks processing timestamp,
#    independent of the source's `_loaded_at` field.
# 4. Retains Kafka metadata columns for offset tracking & debugging.
# 5. Silver layer compatibility preserved: all original business
#    fields remain at the top level with identical names/types.
# ============================================================

bronze_stream_df = (
    raw_stream_df
    .select(
        # --- Kafka metadata for lineage & deduplication ---
        col("topic").alias("_kafka_topic"),
        col("partition").alias("_kafka_partition"),
        col("offset").alias("_kafka_offset"),
        col("timestamp").alias("_kafka_timestamp"),

        # --- Raw payload preserved for replay ---
        col("value").cast("string").alias("_raw_payload"),

        # --- Parsed fields with PERMISSIVE corrupt-record handling ---
        from_json(
            col("value").cast("string"),
            market_schema,
            {"mode": "PERMISSIVE"}  # bad JSON → _corrupt_record, no failure
        ).alias("parsed")
    )
    .select(
        # Kafka metadata
        "_kafka_topic",
        "_kafka_partition",
        "_kafka_offset",
        "_kafka_timestamp",
        "_raw_payload",

        # Business fields (unchanged names for Silver compatibility)
        "parsed.symbol",
        "parsed.asset_class",
        "parsed.trade_id",
        "parsed.trade_price",
        "parsed.trade_quantity",
        "parsed.event_type",
        "parsed.trade_timestamp",
        "parsed._source",
        "parsed._loaded_at",
        "parsed._batch_id",
        "parsed._corrupt_record",

        # Ingestion metadata — when Databricks processed the event
        current_timestamp().alias("_ingestion_ts"),
    )
)

In [0]:
# ============================================================
# STREAM WRITE — Delta with production hardening
# ============================================================
# Improvements:
# 1. trigger(availableNow=True) processes all available data then
#    stops — ideal for scheduled job execution (Lakeflow Jobs).
#    Switch to processingTime="30 seconds" for always-on streaming.
# 2. mergeSchema allows new upstream columns to be added to the
#    Delta table automatically without manual ALTER TABLE.
# 3. queryName identifies the stream in Spark UI / monitoring.
# 4. awaitTermination() blocks until the batch completes, ensuring
#    the job reports success/failure correctly.
# ============================================================

bronze_query = (
    bronze_stream_df.writeStream
        .format("delta")
        .outputMode("append")
        .queryName("bronze_crypto_trades_ingestion")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .option("mergeSchema", "true")       # auto-evolve schema on new fields
        .trigger(availableNow=True)           # batch-style; swap for continuous
        .start(BRONZE_PATH)
)

# Block until micro-batch completes (critical for job orchestration)
bronze_query.awaitTermination()

In [0]:
# ============================================================
# POST-INGESTION VERIFICATION
# ============================================================
# Quick sanity check after each run. In production, wire these
# metrics into your monitoring/alerting system (e.g., Datadog, PagerDuty).
# ============================================================

bronze_table = spark.read.format("delta").load(BRONZE_PATH)

print(f"✓ Bronze table record count: {bronze_table.count():,}")
print(f"✓ Corrupt records: {bronze_table.filter(col('_corrupt_record').isNotNull()).count():,}")

# Show sample including new metadata columns
display(
    bronze_table
        .select(
            "_ingestion_ts", "_kafka_offset", "symbol",
            "trade_price", "trade_quantity", "_corrupt_record"
        )
        .orderBy(col("_ingestion_ts").desc())
        .limit(5)
)